In [1]:
pip install gensim scikit-learn seaborn matplotlib


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
from gensim.models import Word2Vec
import numpy as np
import pandas as pd
import os
import json
from scipy.spatial.distance import cosine
from sklearn.metrics.pairwise import cosine_distances
from scipy.stats import pearsonr
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

In [3]:
pd.set_option('display.max_columns', None)

In [4]:
with open("Data/century_tokens.json", "r", encoding="utf-8") as f:
    century_tokens_loaded = json.load(f)

In [5]:
century_tokens_loaded.keys()

dict_keys(['5', '12', '13', '14', '15', '16', '18', '19', '20'])

In [6]:
century_5th = century_tokens_loaded['5']
century_12th = century_tokens_loaded['12']
century_13th = century_tokens_loaded['13']
century_14th = century_tokens_loaded['14']
century_15th = century_tokens_loaded['15']
century_16th = century_tokens_loaded['16']
century_18th = century_tokens_loaded['18']
century_19th = century_tokens_loaded['19']
century_20th = century_tokens_loaded['20']

In [7]:
century_5th

[['සම්යක්', 'යමු', 'බුද්ධාය'],
 ['කාම',
  'ලොක',
  'රූප',
  'ලොක',
  'අරූපලෝක',
  'යන්',
  'ලොකත්රයට',
  'ශ්රෙෂ්ඨ',
  'වූ',
  'මුනීන්ද්ර',
  'නම්',
  'වූ',
  'සම්යක්',
  'සම්බුද්ධයන්',
  'වහන්සේගේ',
  'ශ්රී',
  'පාද',
  'පද්මය',
  'භය',
  'ලොහ',
  'කුලාචාරයෙන්',
  'විනා',
  'සකසා',
  'වැඳ',
  'සකල',
  'ශාස්ත්රයෙහි',
  'පාරප්රාප්ත',
  'පණ්ඩිත',
  'වූ',
  'භෛෂජ්යාර්ථයෙහි',
  'දක්ෂ',
  'වූ',
  'වෙදුන්',
  'විසින්',
  'උතුම්',
  'වූ',
  'රස',
  'සංහිතාදි',
  'තන්ත්ර',
  'ශාස්ත්රයෙහි',
  'වුත',
  'කියන',
  'ලද',
  'විවිධං',
  'නානාප්රකාර',
  'වූ',
  'සාරාර්ථය',
  'උදුරාගෙණ',
  'සකල',
  'සත්වයන්ගේ',
  'රොග',
  'ව්යපසමන',
  'සංඛ්යාත',
  'සංග්රහව',
  'පිණිස',
  'මේ',
  'සාරාර්ථ',
  'සංග්රහ',
  'නම්',
  'ප්රකරණය',
  'සර්ථාවබොධය',
  'සඳහා',
  'කියන්නෙමි'],
 ['යනාදී',
  'අනෙකවිධ',
  'මානයෙන්',
  'දැක්වූ',
  'මේ',
  'වෛද්ය',
  'ශාස්ත්රය',
  'ප්රථම',
  'කල්පයෙහි',
  'ලෝකාර්ථය',
  'කරන්නා',
  'වූ',
  'මහ',
  'බ්රහ්මයා',
  'දක්ෂ',
  'පුජාපති',
  'හට',
  'කීහ'],
 ['දක්ෂ', 'ප්රජාපති', 'අග', 'විනීන්', '

In [8]:
print(f"""Number of sentences in 5th century: {len(century_5th)}\n
      Number of sentences in 12th century: {len(century_12th)}\n
      Number of sentences in 13th century: {len(century_13th)}\n
      Number of sentences in 14th century: {len(century_14th)}\n
      Number of sentences in 15th century: {len(century_15th)}\n
      Number of sentences in 16th century: {len(century_16th)}\n
      Number of sentences in 18th century: {len(century_18th)}\n
      Number of sentences in 19th century: {len (century_19th)}\n
      Number of sentences in 20th century: {len(century_20th)}""")

Number of sentences in 5th century: 33

      Number of sentences in 12th century: 59

      Number of sentences in 13th century: 523

      Number of sentences in 14th century: 80

      Number of sentences in 15th century: 144

      Number of sentences in 16th century: 10

      Number of sentences in 18th century: 112

      Number of sentences in 19th century: 708

      Number of sentences in 20th century: 822


In [9]:
class CleanCorpusIterator:
    def __init__(self, data):
        self.data = data
        # We pre-calculate this once so it's fast later
        self._length = sum(1 for item in self.data if isinstance(item, list) and item != "__DOC_END__")
        
    def __iter__(self):
        for item in self.data:
            if item == "__DOC_END__":
                continue
            if isinstance(item, list):
                yield item
                
    def __len__(self):
        # This fixes the TypeError
        return self._length

In [10]:
century_18th = CleanCorpusIterator(century_18th)

In [11]:
century_20th = CleanCorpusIterator(century_20th)

In [12]:
# # Exp1
# model_5th = Word2Vec(sentences=century_5th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_12th = Word2Vec(sentences=century_12th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_13th = Word2Vec(sentences=century_13th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_14th = Word2Vec(sentences=century_14th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_15th = Word2Vec(sentences=century_15th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_16th = Word2Vec(sentences=century_16th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_18th = Word2Vec(sentences=century_18th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_19th = Word2Vec(sentences=century_19th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)
# model_20th = Word2Vec(sentences=century_20th, vector_size=100, window=4, min_count=1, workers=1, epochs=10, seed=42)

In [13]:
# # Exp2
# model_5th = Word2Vec(sentences=century_5th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_12th = Word2Vec(sentences=century_12th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_13th = Word2Vec(sentences=century_13th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_14th = Word2Vec(sentences=century_14th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_15th = Word2Vec(sentences=century_15th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_16th = Word2Vec(sentences=century_16th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_18th = Word2Vec(sentences=century_18th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_19th = Word2Vec(sentences=century_19th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)
# model_20th = Word2Vec(sentences=century_20th, vector_size=25, window=15, min_count=1, workers=1, hs=1, epochs=100, seed=42)

In [14]:
from gensim.models.callbacks import CallbackAny2Vec

class LossLogger(CallbackAny2Vec):
    '''Callback to log loss after each epoch.'''
    def __init__(self):
        self.epoch = 0
        self.last_loss = 0

    def on_epoch_end(self, model):
        loss = model.get_latest_training_loss()
        # Calculate loss for the current epoch specifically
        current_loss = loss - self.last_loss
        self.last_loss = loss
        print(f"Epoch {self.epoch} Loss: {current_loss}")
        self.epoch += 1

In [15]:
logging_callback = LossLogger()

In [16]:
model_13th = Word2Vec(sentences=century_13th, vector_size=50,
    window=5,
    min_count=1,
    sg=1,
    hs=0,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 166443.578125
Epoch 1 Loss: 147926.953125
Epoch 2 Loss: 100248.84375
Epoch 3 Loss: 69546.71875
Epoch 4 Loss: 58404.53125
Epoch 5 Loss: 54357.25
Epoch 6 Loss: 53474.375
Epoch 7 Loss: 52810.5625
Epoch 8 Loss: 51550.4375
Epoch 9 Loss: 52124.0625
Epoch 10 Loss: 52346.5
Epoch 11 Loss: 50433.625
Epoch 12 Loss: 52240.0
Epoch 13 Loss: 53494.375
Epoch 14 Loss: 52775.6875


Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Epoch 15 Loss: 51275.5
Epoch 16 Loss: 50565.25
Epoch 17 Loss: 51403.75
Epoch 18 Loss: 53710.375
Epoch 19 Loss: 52138.875
Epoch 20 Loss: 50415.875
Epoch 21 Loss: 49786.625
Epoch 22 Loss: 49163.125
Epoch 23 Loss: 51551.0
Epoch 24 Loss: 51652.125
Epoch 25 Loss: 49218.5
Epoch 26 Loss: 49844.375
Epoch 27 Loss: 51868.5
Epoch 28 Loss: 51586.0
Epoch 29 Loss: 47355.875
Epoch 30 Loss: 49333.75
Epoch 31 Loss: 48643.75
Epoch 32 Loss: 49095.875
Epoch 33 Loss: 48936.0
Epoch 34 Loss: 46186.5
Epoch 35 Loss: 41981.125
Epoch 36 Loss: 37139.5
Epoch 37 Loss: 36385.5
Epoch 38 Loss: 35840.75
Epoch 39 Loss: 32520.0
Epoch 40 Loss: 32032.25
Epoch 41 Loss: 32008.5
Epoch 42 Loss: 29012.75
Epoch 43 Loss: 27789.75
Epoch 44 Loss: 26018.25
Epoch 45 Loss: 24496.75
Epoch 46 Loss: 23562.5
Epoch 47 Loss: 23012.5
Epoch 48 Loss: 21386.25
Epoch 49 Loss: 21472.25
Epoch 50 Loss: 20362.25
Epoch 51 Loss: 18618.25
Epoch 52 Loss: 17204.75
Epoch 53 Loss: 17734.5
Epoch 54 Loss: 17177.5
Epoch 55 Loss: 16480.75
Epoch 56 Loss: 16479.

In [17]:
logging_callback = LossLogger()

In [18]:
model_14th = Word2Vec(sentences=century_14th, vector_size=50,
    window=5,
    min_count=1,
    sg=1,
    hs=0,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 56017.25
Epoch 1 Loss: 55705.921875
Epoch 2 Loss: 54613.25
Epoch 3 Loss: 53265.265625
Epoch 4 Loss: 45684.15625
Epoch 5 Loss: 33623.65625
Epoch 6 Loss: 27005.0
Epoch 7 Loss: 24814.09375
Epoch 8 Loss: 22658.375
Epoch 9 Loss: 20179.59375
Epoch 10 Loss: 19988.65625
Epoch 11 Loss: 18247.03125
Epoch 12 Loss: 18573.5625
Epoch 13 Loss: 19471.09375
Epoch 14 Loss: 18251.71875
Epoch 15 Loss: 18859.375
Epoch 16 Loss: 18000.875
Epoch 17 Loss: 19037.6875
Epoch 18 Loss: 17727.5
Epoch 19 Loss: 18717.6875
Epoch 20 Loss: 18583.5
Epoch 21 Loss: 18146.0
Epoch 22 Loss: 17588.75
Epoch 23 Loss: 17299.9375
Epoch 24 Loss: 17312.1875
Epoch 25 Loss: 17114.75
Epoch 26 Loss: 17000.6875
Epoch 27 Loss: 17506.125
Epoch 28 Loss: 17439.125
Epoch 29 Loss: 18150.0
Epoch 30 Loss: 17968.875
Epoch 31 Loss: 18124.625
Epoch 32 Loss: 17755.3125
Epoch 33 Loss: 17656.625
Epoch 34 Loss: 17390.875
Epoch 35 Loss: 18506.6875
Epoch 36 Loss: 16613.0
Epoch 37 Loss: 17806.1875
Epoch 38 Loss: 18353.5
Epoch 39 Loss: 17589.7

In [19]:
logging_callback = LossLogger()

In [20]:
model_15th = Word2Vec(sentences=century_15th, vector_size=50,
    window=5,
    min_count=1,
    sg=1,
    hs=0,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 124295.65625
Epoch 1 Loss: 119579.046875
Epoch 2 Loss: 114364.640625
Epoch 3 Loss: 93041.46875
Epoch 4 Loss: 75053.75
Epoch 5 Loss: 52387.0625
Epoch 6 Loss: 45345.375
Epoch 7 Loss: 44033.875
Epoch 8 Loss: 42147.125
Epoch 9 Loss: 41917.6875
Epoch 10 Loss: 39456.9375
Epoch 11 Loss: 39252.75
Epoch 12 Loss: 37689.75
Epoch 13 Loss: 39588.5
Epoch 14 Loss: 40082.5
Epoch 15 Loss: 38483.9375
Epoch 16 Loss: 39405.8125
Epoch 17 Loss: 38349.0
Epoch 18 Loss: 38005.375
Epoch 19 Loss: 36370.25
Epoch 20 Loss: 38539.25
Epoch 21 Loss: 36577.875
Epoch 22 Loss: 36519.125
Epoch 23 Loss: 36157.375
Epoch 24 Loss: 38206.125
Epoch 25 Loss: 38981.75
Epoch 26 Loss: 38227.0
Epoch 27 Loss: 35803.25
Epoch 28 Loss: 38241.125
Epoch 29 Loss: 38578.625
Epoch 30 Loss: 38164.125
Epoch 31 Loss: 38997.625
Epoch 32 Loss: 36366.75
Epoch 33 Loss: 37944.0
Epoch 34 Loss: 36816.5
Epoch 35 Loss: 37430.625
Epoch 36 Loss: 37314.5
Epoch 37 Loss: 37523.875
Epoch 38 Loss: 37361.625
Epoch 39 Loss: 36649.625
Epoch 40 Loss:

In [21]:
logging_callback = LossLogger()

In [22]:
model_18th = Word2Vec(sentences=century_18th, vector_size=50,
    window=5,
    min_count=1,
    sg=1,
    hs=0,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 30520.02734375
Epoch 1 Loss: 30656.72265625
Epoch 2 Loss: 31105.2265625
Epoch 3 Loss: 33938.2734375
Epoch 4 Loss: 32082.5
Epoch 5 Loss: 27595.875
Epoch 6 Loss: 23019.234375
Epoch 7 Loss: 18046.1875
Epoch 8 Loss: 17284.28125
Epoch 9 Loss: 14681.640625
Epoch 10 Loss: 13515.34375
Epoch 11 Loss: 13349.625
Epoch 12 Loss: 13269.0625
Epoch 13 Loss: 11907.34375
Epoch 14 Loss: 11443.90625
Epoch 15 Loss: 10941.46875
Epoch 16 Loss: 10299.78125
Epoch 17 Loss: 9776.1875
Epoch 18 Loss: 10931.46875
Epoch 19 Loss: 10918.6875
Epoch 20 Loss: 11066.9375
Epoch 21 Loss: 10098.71875
Epoch 22 Loss: 10854.40625
Epoch 23 Loss: 10295.75
Epoch 24 Loss: 10691.09375
Epoch 25 Loss: 11767.59375
Epoch 26 Loss: 10340.625
Epoch 27 Loss: 9839.25
Epoch 28 Loss: 11196.8125
Epoch 29 Loss: 11251.78125
Epoch 30 Loss: 10133.4375
Epoch 31 Loss: 11490.25
Epoch 32 Loss: 10349.3125
Epoch 33 Loss: 9685.25
Epoch 34 Loss: 9673.75
Epoch 35 Loss: 10317.5625
Epoch 36 Loss: 9934.625
Epoch 37 Loss: 10702.75
Epoch 38 Loss: 1

In [23]:
logging_callback = LossLogger()

In [24]:
model_19th = Word2Vec(sentences=century_19th, vector_size=50,
    window=5,
    min_count=1,
    sg=1,
    hs=0,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 137996.328125
Epoch 1 Loss: 142773.390625
Epoch 2 Loss: 124102.84375
Epoch 3 Loss: 89202.625
Epoch 4 Loss: 65196.4375
Epoch 5 Loss: 53105.0625
Epoch 6 Loss: 47909.375
Epoch 7 Loss: 46054.0625
Epoch 8 Loss: 44846.4375
Epoch 9 Loss: 43640.0
Epoch 10 Loss: 44028.25
Epoch 11 Loss: 44494.75
Epoch 12 Loss: 42550.875
Epoch 13 Loss: 43733.25
Epoch 14 Loss: 44547.1875
Epoch 15 Loss: 43721.75
Epoch 16 Loss: 40843.125
Epoch 17 Loss: 42645.25
Epoch 18 Loss: 42182.5
Epoch 19 Loss: 41666.875
Epoch 20 Loss: 41068.875
Epoch 21 Loss: 40146.0
Epoch 22 Loss: 41966.75
Epoch 23 Loss: 40099.875
Epoch 24 Loss: 40031.0
Epoch 25 Loss: 41192.375
Epoch 26 Loss: 40419.875
Epoch 27 Loss: 41491.625
Epoch 28 Loss: 41669.25
Epoch 29 Loss: 39827.125
Epoch 30 Loss: 42282.0
Epoch 31 Loss: 40506.5
Epoch 32 Loss: 40216.375
Epoch 33 Loss: 38975.75
Epoch 34 Loss: 40617.875
Epoch 35 Loss: 39724.625
Epoch 36 Loss: 39836.0
Epoch 37 Loss: 40167.75
Epoch 38 Loss: 39543.5
Epoch 39 Loss: 38868.5
Epoch 40 Loss: 39581.

In [25]:
logging_callback = LossLogger()

In [26]:
model_20th = Word2Vec(sentences=century_20th, vector_size=50,
    window=5,
    min_count=1,
    sg=1,
    hs=0,
    negative=15,
    sample=1e-5,
    alpha=0.05,
    min_alpha=0.0007,
    workers=4,
    compute_loss=True, # MUST be True
    callbacks=[logging_callback],
    epochs=200,
    seed=42)

Epoch 0 Loss: 174628.484375
Epoch 1 Loss: 176341.421875
Epoch 2 Loss: 132280.84375
Epoch 3 Loss: 88044.9375
Epoch 4 Loss: 68502.0
Epoch 5 Loss: 58029.75
Epoch 6 Loss: 59214.0625
Epoch 7 Loss: 56730.8125
Epoch 8 Loss: 54763.25
Epoch 9 Loss: 56549.4375
Epoch 10 Loss: 57434.6875
Epoch 11 Loss: 55085.5
Epoch 12 Loss: 56405.8125
Epoch 13 Loss: 52826.125
Epoch 14 Loss: 54122.5
Epoch 15 Loss: 54773.25
Epoch 16 Loss: 54742.875
Epoch 17 Loss: 54722.625
Epoch 18 Loss: 54125.625
Epoch 19 Loss: 53965.625
Epoch 20 Loss: 53533.0
Epoch 21 Loss: 54458.375
Epoch 22 Loss: 53853.5
Epoch 23 Loss: 52807.375
Epoch 24 Loss: 54236.125
Epoch 25 Loss: 54386.75
Epoch 26 Loss: 53720.375
Epoch 27 Loss: 51969.0
Epoch 28 Loss: 53716.375
Epoch 29 Loss: 54686.375
Epoch 30 Loss: 52546.0
Epoch 31 Loss: 50347.875
Epoch 32 Loss: 44629.0
Epoch 33 Loss: 44285.0
Epoch 34 Loss: 46065.5
Epoch 35 Loss: 42760.25
Epoch 36 Loss: 43516.75
Epoch 37 Loss: 41668.0
Epoch 38 Loss: 41013.0
Epoch 39 Loss: 38908.25
Epoch 40 Loss: 38176.5
E

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


Epoch 101 Loss: 8978.75
Epoch 102 Loss: 9165.5
Epoch 103 Loss: 8883.75
Epoch 104 Loss: 9046.25
Epoch 105 Loss: 8822.0
Epoch 106 Loss: 9481.0
Epoch 107 Loss: 9082.25
Epoch 108 Loss: 8991.0
Epoch 109 Loss: 8926.25
Epoch 110 Loss: 8519.25
Epoch 111 Loss: 8794.0
Epoch 112 Loss: 9138.25
Epoch 113 Loss: 8813.0
Epoch 114 Loss: 8566.75
Epoch 115 Loss: 8642.5
Epoch 116 Loss: 8276.75
Epoch 117 Loss: 8374.0
Epoch 118 Loss: 8894.5
Epoch 119 Loss: 7973.25
Epoch 120 Loss: 8073.75
Epoch 121 Loss: 8475.75
Epoch 122 Loss: 8600.75
Epoch 123 Loss: 8340.25
Epoch 124 Loss: 8457.5
Epoch 125 Loss: 8732.0
Epoch 126 Loss: 7964.25
Epoch 127 Loss: 8572.25
Epoch 128 Loss: 8411.25
Epoch 129 Loss: 7957.25
Epoch 130 Loss: 8793.75
Epoch 131 Loss: 8196.5
Epoch 132 Loss: 8655.5
Epoch 133 Loss: 7764.25
Epoch 134 Loss: 8253.5
Epoch 135 Loss: 8512.0
Epoch 136 Loss: 7846.0
Epoch 137 Loss: 7792.75
Epoch 138 Loss: 7494.75
Epoch 139 Loss: 8088.5
Epoch 140 Loss: 7942.0
Epoch 141 Loss: 7467.25
Epoch 142 Loss: 7585.75
Epoch 143 

In [27]:
logging_callback = LossLogger()

In [28]:
model_13th.save("Word2Vec/Exp3/model_13th.model")
model_14th.save("Word2Vec/Exp3/model_14th.model")
model_15th.save("Word2Vec/Exp3/model_15th.model")
model_18th.save("Word2Vec/Exp3/model_18th.model")
model_19th.save("Word2Vec/Exp3/model_19th.model")
model_20th.save("Word2Vec/Exp3/model_20th.model")

In [29]:
# all_centuries_data = []
# for century in century_tokens_loaded.values():
#     all_centuries_data.extend(century)

In [30]:
# all_centuries_data = CleanCorpusIterator(all_centuries_data)

In [31]:
# base_model = Word2Vec(
#     vector_size=50,
#     window=8,
#     min_count=1,
#     sg=1,
#     hs=0,
#     negative=20,
#     sample=1e-2,
#     alpha=0.03,
#     min_alpha=0.0007,
#     workers=4,
#     seed=42
# )

# # 2. Build the Vocabulary (Tells the model which words exist)
# base_model.build_vocab(all_centuries_data)

# # 3. Train the Model (Learn the relationships)
# base_model.train(
#     all_centuries_data, 
#     total_examples=base_model.corpus_count, 
#     epochs=50 # Use the full 50 here
# )

In [32]:
# print(f"Total words in vocabulary: {len(base_model.wv)}")
# # Check a few neighbors to see if it makes sense for your dataset
# print(base_model.wv.most_similar(positive=['රජ'], topn=5))

In [33]:
print(f"Total words in vocabulary: {len(model_20th.wv)}")
# Check a few neighbors to see if it makes sense for your dataset
print(model_20th.wv.most_similar(positive=['රජ'], topn=5))

Total words in vocabulary: 7993
[('මත්රියනී', 0.6849343180656433), ('රකී', 0.6811801195144653), ('පහදා', 0.6724487543106079), ('වවණේ', 0.6604266166687012), ('බැදී', 0.6463024616241455)]


In [34]:
# base_model.train(century_20th, total_examples=len(century_20th), epochs=100)

In [35]:
# model_20th.train(
#     century_20th, 
#     total_examples=len(century_20th), 
#     epochs=50 # Moderate fine-tuning
# )

In [36]:
# model_base = Word2Vec(
#     sentences=all_centuries_data, 
#     vector_size=100, 
#     window=5, 
#     min_count=1, 
#     sg=1, 
#     workers=4, 
#     epochs=20 # Enough to learn general structure
# )
# model_base.save("base_foundation.model")

In [37]:
centuries = ['13th', '14th', '15th', '18th', '19th', '20th']

In [38]:
import io
import os

# Ensure the directory exists
os.makedirs('Word2Vec/Embeddings', exist_ok=True)

for c_suffix in centuries:

    model_variable_name = f'model_{c_suffix}'
    
    if model_variable_name not in globals():
        print(f"Skipping {model_variable_name}: Model object not found.")
        continue
        
    current_model = globals()[model_variable_name]
    
    save_path = f'Word2Vec/Embeddings/{c_suffix}'
    os.makedirs(save_path, exist_ok=True)

    out_v = io.open(f'{save_path}/vectors.tsv', 'w', encoding='utf-8')
    out_m = io.open(f'{save_path}/metadata.tsv', 'w', encoding='utf-8')

    current_model.wv.fill_norms() 
    words = current_model.wv.index_to_key

    for word in words:
        vec = current_model.wv.get_vector(word, norm=True)
        
        out_v.write('\t'.join([str(x) for x in vec]) + "\n")
        out_m.write(word + "\n")

    out_v.close()
    out_m.close()
    print(f"Successfully exported {model_variable_name}")

Successfully exported model_13th
Successfully exported model_14th
Successfully exported model_15th
Successfully exported model_18th
Successfully exported model_19th
Successfully exported model_20th
